## 🏗️ The System Architecture

To map multiple images to multiple historical text reports per patient (as structured in your CSV), you need an encoder-decoder setup:

1. **Vision Encoder:** A convolutional network (like ResNet or DenseNet) or a Vision Transformer (ViT) that extracts deep spatial features from the chest X-rays.
2. **Text Decoder:** A causal language model (like GPT-2, DistilGPT2, or a lightweight LLaMA variant) that takes the image features and autoregressively generates the clinical report text tokens.

```
[ Input Images ] ───► [ Vision Encoder (e.g., ViT) ] ───┐
                                                         ▼
[ Ground Truth ] ───► [ Text Decoder (e.g., GPT-2) ] ──► [ Generated Report ]

```

In [13]:
import os
import ast
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from torchvision import transforms, models
from transformers import GPT2LMHeadModel, GPT2Config, AutoTokenizer
from tqdm import tqdm

## 🛠️ Data Preprocessing & Custom Dataset

Because your CSV stores paths and text as string representations of lists, your PyTorch `Dataset` needs to parse these strings, load the actual image files, tokenize the reports, and align them.

Here is how you can build the data pipeline:

In [2]:
class MimicReportDataset(Dataset):
    def __init__(self, csv_file, img_root_dir, tokenizer, transform=None, max_length=128):
        """
        Args:
            csv_file (str): Path to your processed CSV file.
            img_root_dir (str): Root directory where the 'files/' folder is located.
            tokenizer: Hugging Face tokenizer (e.g., GPT2Tokenizer).
        """
        self.df = pd.read_csv(csv_file)
        self.img_root_dir = img_root_dir
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Standard image transformations for DenseNet121
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Parse string representation of lists safely using abstract syntax trees
        img_paths = ast.literal_eval(row['image'])
        reports = ast.literal_eval(row['text'])
        
        # 2. Extract the first available image and its matching clinical report text
        img_path = os.path.join(self.img_root_dir, img_paths[0])
        report_text = reports[0] if len(reports) > 0 else "Findings: Normal study."
        
        # 3. Load and transform image file safely
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
        except Exception as e:
            # Fallback tensor if a file is missing or corrupted during runtime
            image = torch.zeros(3, 224, 224)
            
        # 4. Tokenize report text for GPT-2
        tokens = self.tokenizer(
            report_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)
        
        # Create shifting target labels for text generation loss calculation
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'image': image,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

## 🏗️ Define the Multimodal Model

This model extracts features from the image, projects them into the text dimension, and prepends them as visual "tokens" right before the actual report text tokens.

In [3]:
class MedicalReportGenerator(nn.Module):
    def __init__(self, gpt2_model_name="gpt2", embed_dim=768):
        super(MedicalReportGenerator, self).__init__()
        
        # 1. Vision Encoder: Using pre-trained DenseNet121
        # Excellent for chest X-rays because of feature reuse dense connections
        self.encoder = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.encoder.classifier.in_features
        self.encoder.classifier = nn.Identity() # Remove the final classification layer
        
        # 2. Projection Layer: Maps vision features (1024) to Text Embeddings (768)
        self.projector = nn.Linear(num_ftrs, embed_dim)
        
        # 3. Text Decoder: Causal Language Model
        self.decoder = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        
    def forward(self, images, input_ids, attention_mask):
        # Step 1: Extract visual features -> Shape: [batch_size, 1024]
        visual_features = self.encoder(images)
        
        # Step 2: Project to text embedding space -> Shape: [batch_size, 768]
        image_embeddings = self.projector(visual_features)
        # Add a pseudo-sequence dimension -> Shape: [batch_size, 1, 768]
        image_embeddings = image_embeddings.unsqueeze(1) 
        
        # Step 3: Get standard text token embeddings -> Shape: [batch_size, seq_len, 768]
        text_embeddings = self.decoder.transformer.wte(input_ids)
        
        # Step 4: Concatenate visual "tokens" with actual text tokens
        # Shape becomes: [batch_size, 1 + seq_len, 768]
        multimodal_embeddings = torch.cat((image_embeddings, text_embeddings), dim=1)
        
        # Adjust attention mask to account for the extra visual token at the start
        batch_size = images.size(0)
        extra_mask = torch.ones((batch_size, 1), device=images.device)
        extended_attention_mask = torch.cat((extra_mask, attention_mask), dim=1)
        
        # Step 5: Pass through GPT-2 layers
        outputs = self.decoder(inputs_embeds=multimodal_embeddings, attention_mask=extended_attention_mask)
        return outputs.logits

## 🏎️ The Efficient Strategy: Progressive Unfreezing

To train this efficiently without destroying the pre-trained weights (and saving hours of compute), we will freeze the encoder first, train the mapping, and unfreeze later.


In [4]:
def get_trainable_parameters(model, phase="warmup"):
    if phase == "warmup":
        print("🔥 Phase 1: Freezing Encoder. Training Projector and Decoder Only.")
        # Freeze DenseNet
        for param in model.encoder.parameters():
            param.requires_grad = False
        # Keep Projector and GPT-2 Trainable
        for param in model.projector.parameters():
            param.requires_grad = True
        for param in model.decoder.parameters():
            param.requires_grad = True
            
    elif phase == "joint":
        print("🔓 Phase 2: Unfreezing Everything for joint fine-tuning.")
        for param in model.parameters():
            param.requires_grad = True
            
    return filter(lambda p: p.requires_grad, model.parameters())

## 🔄 PyTorch Training Loop Execution

This standard PyTorch loop tracks training by passing shifted tokens into the standard Cross-Entropy loss layer—meaning the model constantly tries to predict the next word in the findings section based on the image features it calculated.


In [5]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass
        logits = model(images, input_ids, attention_mask)
        
        # Real-time shifting for standard Autoregressive Generation Loss
        # We target predictions on text tokens (ignoring the visual prefix token projection)
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        # Flatten tensors for CrossEntropy Loss calculation
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

## 🔮 Generation Pipeline (Inference)

When running inference on new X-ray images, the model uses an autoregressive decoding loop to generate word-by-word descriptions until it hits an `<EOS>` (end-of-sentence) token.

In [6]:
def generate_report(model, image, tokenizer, device, max_len=64):
    model.eval()
    with torch.no_grad():
        image = image.to(device).unsqueeze(0) # Add batch dimension
        
        # Initialize text generation with the Start of Text Token (BOS)
        generated = torch.tensor([[tokenizer.bos_token_id]]).to(device)
        attention_mask = torch.ones((1, 1)).to(device)
        
        for _ in range(max_len):
            logits = model(image, generated, attention_mask)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
            
            # Append predicted token to sequence
            generated = torch.cat((generated, next_token), dim=1)
            attention_mask = torch.cat((attention_mask, torch.ones((1, 1)).to(device)), dim=1)
            
            # Break loop early if model outputs End of Text Token
            if next_token.item() == tokenizer.eos_token_id:
                break
                
        return tokenizer.decode(generated[0], skip_special_tokens=True)



## 🛠️ Execution Checklist to Run This Code:

1. **Instantiation:** 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedicalReportGenerator().to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

2. **Warmup Phase (approx. 3-5 epochs):** Run `get_trainable_parameters(model, phase="warmup")` with a learning rate of `1e-4`. This aligns your projection layer quickly.
3. **Joint Phase (approx. 5 epochs):** Run `get_trainable_parameters(model, phase="joint")` with a very low learning rate (`1e-5`) to optimize details across the entire network architecture concurrently.

## 🛠️ Connect the Dataset to Your Local Paths

Paste this cell into your notebook right after your model definition. We will point the `img_root_dir` exactly to where your `files` folder is hosted.

In [ ]:
# 1. Initialize the tokenizer (matches our GPT-2 decoder)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. Define the absolute or relative paths based on your workspace
# Your CSV files are right next to the notebook, and the images are inside the subfolder
IMG_ROOT = "MIMC-CXR/official_data_iccv_final"
TRAIN_CSV = "mimic_cxr_aug_train.csv"
VALID_CSV = "mimic_cxr_aug_validate.csv"

# 3. Create Train and Validation Datasets
# (Uses the MimicReportDataset class we created earlier)
train_dataset = MimicReportDataset(
    csv_file=TRAIN_CSV,
    img_root_dir=IMG_ROOT,
    tokenizer=tokenizer,
    max_length=128
)

val_dataset = MimicReportDataset(
    csv_file=VALID_CSV,
    img_root_dir=IMG_ROOT,
    tokenizer=tokenizer,
    max_length=128
)

# 4. Create DataLoaders
# Pin memory speeds up tensor transfer to your GPU/MPS backend
BATCH_SIZE = 8  # Adjust based on your available VRAM

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(size_check := f"Data pipelines ready! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Data pipelines ready! Train batches: 8074 | Val batches: 63


## 🛠️ Step 2: Test the Pipeline with a Single Sample

Before running a heavy training loop, it is critical to run a sanity check. This verifies that Python is reading your `.jpg` paths correctly out of the directory structure and tokenizing the text without throwing any path mismatches.

In [12]:
# Pull a single batch from your train loader
try:
    sample_batch = next(iter(train_loader))
    print("✅ Success! Images and text loaded perfectly.")
    print("Image Tensor Shape:", sample_batch['image'].shape)        # Expected: [8, 3, 224, 224]
    print("Input IDs Tensor Shape:", sample_batch['input_ids'].shape) # Expected: [8, 128]
except Exception as e:
    print("❌ There is an issue with pathing or parsing:")
    print(str(e))

✅ Success! Images and text loaded perfectly.
Image Tensor Shape: torch.Size([8, 3, 224, 224])
Input IDs Tensor Shape: torch.Size([8, 128])


## 🛠️ Step 3: Initialize the Warmup Phase Training

Once your sample batch passes without an error, you can initialize the optimizer and run your first training epoch. Because you're working in a `.ipynb` notebook, let's wrap the training execution nicely using the `tqdm` progress bar you verified earlier.

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedicalReportGenerator().to(device)

# Activate Phase 1: Freeze encoder weights, train projector and decoder only
trainable_params = get_trainable_parameters(model, phase="warmup")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

# Simple loop to run 1 epoch
model.train()
epoch_loss = 0

# The progress bar will now render nicely text-based or via interactive widgets
progress_bar = tqdm(train_loader, desc="Training Epoch 1")

for batch in progress_bar:
    images = batch['image'].to(device)
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    optimizer.zero_grad()
    
    # Forward pass
    logits = model(images, input_ids, attention_mask)
    
    # Shift outputs to calculate causal language loss
    shift_logits = logits[:, 1:-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    
    loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item()
    progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

print(f"Epoch 1 Completed. Average Loss: {epoch_loss / len(train_loader):.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

🔥 Phase 1: Freezing Encoder. Training Projector and Decoder Only.


Training Epoch 1: 100%|██████████| 8074/8074 [2:58:04<00:00,  1.32s/it, batch_loss=0.9788]  

Epoch 1 Completed. Average Loss: 1.1568


In [15]:
# Define the file path for your saved checkpoint
CHECKPOINT_PATH = "mimic_vlm_epoch1_checkpoint.pt"

# Create a dictionary containing all vital structural states
checkpoint_data = {
    'epoch': 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 1.1568,
}

# Save securely to disk
torch.save(checkpoint_data, CHECKPOINT_PATH)
print(f"✅ Success! Checkpoint saved as '{CHECKPOINT_PATH}'")
print("You can now safely shut down this notebook and transfer the file to your Windows machine.")

✅ Success! Checkpoint saved as 'mimic_vlm_epoch1_checkpoint.pt'
You can now safely shut down this notebook and transfer the file to your Windows machine.


In [19]:
model.eval()
val_loss = 0

# Disable gradient calculations to save VRAM and speed up validation
with torch.no_grad():
    val_progress = tqdm(val_loader, desc="Validating Model")
    
    for batch in val_progress:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        logits = model(images, input_ids, attention_mask)
        
        # Shift tokens for causal language loss calculation
        shift_logits = logits[:, 1:-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        val_loss += loss.item()
        
        val_progress.set_postfix({"val_batch_loss": f"{loss.item():.4f}"})

final_val_loss = val_loss / len(val_loader)
print(f"\n✨ Validation Complete! Average Validation Loss: {final_val_loss:.4f}")


# --- 2. FINAL SAVE SNAPSHOT ---
FINAL_CHECKPOINT_PATH = f"mimic_vlm_epoch1_checkpoint.pt"

torch.save({
    'epoch': 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': epoch_loss / len(train_loader),
    'val_loss': final_val_loss
}, FINAL_CHECKPOINT_PATH)

print(f"💾 Absolute final checkpoint saved successfully to: {FINAL_CHECKPOINT_PATH}")
print("🎉 All tasks completed successfully. You can safely go to sleep now!")

Validating Model: 100%|██████████| 63/63 [00:26<00:00,  2.37it/s, val_batch_loss=1.4857]



✨ Validation Complete! Average Validation Loss: 1.0262
💾 Absolute final checkpoint saved successfully to: mimic_vlm_epoch1_checkpoint.pt
🎉 All tasks completed successfully. You can safely go to sleep now!
